In [0]:
%run ../config/config

In [0]:
import pandas as pd
import json
import os
import sys
import time
from pyspark.sql.functions import col
from pyspark.sql import F

sys.path.insert(0, str(project_path))
from utils.certification import certify_scoring
from utils.certification_report_generator import CertificationReportGenerator
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

# spark = SparkSession.builder.getOrCreate()

In [0]:
# Logger certification
logger = MLOpsLogger(
    name='07_certification',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '07_certification'
    }
)

In [0]:
# Certification thresholds
certification_thresholds = CERTIFICATION_THRESHOLDS
TABLE_TOLERANCE_PCT = 1.0
TOLERANCE = 0.05
logger.info(f"Configuración de certificación cargada dinámicamente {COMPARE_COLUMNS}")

In [0]:
try:
    logger.info("=" * 60)
    logger.info(f"CERTIFICACIÓN - {env.upper()} - Mes {mes_vta}")
    logger.info("=" * 60)

    start_time = time.time()

    # =============================================================================
    # 1. DETERMINAR AMBIENTES A COMPARAR
    # =============================================================================
    # dev: compara Python (output_table) vs SAS (SCORING_TABLE_SAS)
    # stg: compara STG vs DEV
    # prod: compara PROD vs STG

    if env.lower() == 'dev':
        env_source = 'dev'
        env_target = 'sas'
        scoring_path_source = output_table  # tabla generada por 05_inference
        scoring_path_target = SCORING_TABLE_SAS
    elif env.lower() == 'stg':
        env_source = 'stg'
        env_target = 'dev'
        scoring_path_source = output_table
        scoring_path_target = SCRORING_TABLE_SAS
    elif env.lower() == 'prod':
        env_source = 'prod'
        env_target = 'stg'
        scoring_path_source = output_table
        scoring_path_target = output_table
    else:
        raise ValueError(f"Ambiente no válido: {env}. Use 'dev', 'stg' o 'prod'")

    logger.info(f"Comparación: {env_source.upper()} vs {env_target.upper()}")
    logger.info(f"  Source: {scoring_path_source}")
    logger.info(f"  Target: {scoring_path_target}")

    # =============================================================================
    # 2. CARGAR DATOS
    # =============================================================================
    logger.info(f"\nCargando datos para codmes={mes_vta}...")

    # — Aplanar el struct de variables del modelo ————————————————————————————
    # Si las variables vienen embebidas en el struct "colvariablesmodeloanalitico"
    # (como en la LHCL), se aplanan a columnas planas para comparar cada variable
    # y su valor individualmente (igual que en el reporte de calidad). Si la tabla
    # ya viene plana, se deja tal cual.
    from pyspark.sql.types import StructType as _StructType
    def _aplanar_struct(df, candidatos=("colvariablesmodeloanalitico", "codvariablesmodeloanalitico")):
        _tipos = {f.name: f.dataType for f in df.schema.fields}
        for struct_col in candidatos:
            if struct_col in df.columns and isinstance(_tipos.get(struct_col), _StructType):
                root_cols = [c for c in df.columns if c != struct_col]
                df = df.select(*root_cols, F.col(f"{struct_col}.*"))
                logger.info(f" ✓ Struct '{struct_col}' aplanado: {len(df.columns)} columnas")
                break
        return df

    df_source = spark.read.format("delta").table(scoring_path_source) \
        .filter(col("codmes") == mes_vta)

    df_source = _aplanar_struct(df_source)
    count_source = df_source.count()
    logger.info(f" {env_source.upper()}: {count_source:,} registros")

    if isinstance(scoring_path_target, str):
        df_target = spark.read.format("delta").table(scoring_path_target) \
            .filter(col("codmes") == mes_vta)
    else:
        df_target = scoring_path_target \
            .filter(col("codmes") == mes_vta)

    df_target = df_target.withColumn("numscore", F.round(F.col("numscore"), 0))

    df_target = _aplanar_struct(df_target)
    count_target = df_target.count()
    logger.info(f" {env_target.upper()}: {count_target:,} registros")

    if count_source == 0:
        raise ValueError(f"No hay datos en {env_source.upper()} para mes_vta={mes_vta}")

    if count_target == 0:
        logger.warning(f"⚠️ No hay datos en {env_target.upper()} para mes_vta={mes_vta}")

    # =============================================================================
    # 3. EJECUTAR CERTIFICACIÓN
    # =============================================================================
    logger.info(f"\nEjecutando certificación...")

    certification_thresholds = CERTIFICATION_THRESHOLDS

    def _run_cert(df_s, df_t):
        return certify_scoring(
            df_scoring_source=df_s,
            df_scoring_target=df_t,
            env_source=env_source,
            env_target=env_target,
            join_keys=JOIN_KEYS,
            compare_columns=COMPARE_COLUMNS,
            critical_fields=CRITICAL_FIELDS,
            exclude_columns=EXCLUDE_COLUMNS,
            thresholds=certification_thresholds,
            tolerance=TOLERANCE,
            table_tolerance_pct=TABLE_TOLERANCE_PCT if 'TABLE_TOLERANCE_PCT' in locals() else 1.0
        )

    # Segmentacion (igual que en el reporte de calidad): "Global" (poblacion
    # total) siempre se procesa. Si USE_SEGMENTATION esta activa y la columna de
    # segmentacion existe en AMBAS tablas, se anaden los segmentos de negocio
    # definidos en QUALITY_SEGMENTS. El estado global gobierna el gate.

    _seg_col = globals().get('SEGMENTATION_COLUMN')
    _seg_map = globals().get('QUALITY_SEGMENTS', {}) or {}
    _use_seg = globals().get('USE_SEGMENTATION', False)

    cert_segments = [('Global', df_source, df_target)]

    if _use_seg and _seg_col and _seg_map and _seg_col in df_source.columns and _seg_col in df_target.columns:
        logger.info(f"Segmentacion activa por '{_seg_col}': {list(_seg_map.keys())}")
        for _seg_name, _seg_val in _seg_map.items():
            cert_segments.append(
                _seg_name,
                df_source.filter(col(_seg_col) == _seg_val),
                df_target.filter(col(_seg_col) == _seg_val)
            )
    elif _use_seg and _seg_col and (_seg_col not in df_source.columns or _seg_col not in df_target.columns):
        logger.warning(f"Segmentacion activa pero '{_seg_col}' no existe en ambas tablas; solo 'Global'.")

    segment_reports = []
    for _seg_name, _df_s, _df_t in cert_segments:
        logger.info(f"Certificando segmento: {_seg_name}")
        try:
            segment_reports.append((_seg_name, _run_cert(_df_s, _df_t)))
        except Exception as _seg_err:
            logger.warning(f"Segmento '{_seg_name}' no se pudo certificar: {_seg_err}")

    # El estado global (poblacion total) gobierna el gate de la certificacion.
    cert_report = segment_reports[0][1]
    results = cert_report.get_results()
    summary = results['summary']

    logger.info(f"\n{'=' * 60}")
    logger.info("RESULTADOS")
    logger.info(f"{'=' * 60}")
    logger.info(f"Estado: {summary['overall_status']}")
    logger.info(f"Exitosas: {summary['passed']} | Fallidas: {summary['failed']} | Advertencias: {summary['warnings']}")

    # =============================================================================
    # 4. GENERAR REPORTE HTML
    # =============================================================================
    logger.info(f"\nGenerando reporte HTML...")

    report_generator = CertificationReportGenerator(thresholds=certification_thresholds)
    os.makedirs(temp_reports_certification_path, exist_ok=True)

    report_output_path = os.path.join(
        temp_reports_certification_path,
        f"certification_{env_source}_vs_{env_target}_{mes_vta}.html"
    )

    report_generator.generate_html(
        output_path=report_output_path,
        mes_vta=mes_vta,
        scoring_table=output_table.split('.')[-1],
        input_table=table_name_mt.split('.')[-1],
        source_table=scoring_path_source,
        target_table=scoring_path_target,
        join_keys=JOIN_KEYS,
        segments=segment_reports
    )

    logger.info(f"✓ Reporte generado: {report_output_path}")

    # =============================================================================
    # 5. GUARDAR RESULTADOS
    # =============================================================================
    total_duration = time.time() - start_time

    certification = summary['overall_status'] == 'SUCCESS'
    dbutils.jobs.taskValues.set(key="certification", value=certification)
    dbutils.jobs.taskValues.set(key="report_path", value=report_output_path)
    dbutils.jobs.taskValues.set(key="comparison", value=f"{env_source}_vs_{env_target}")

    if certification:
        logger.info(f"\nCertificación EXITOSA en {total_duration:.2f}s")
    else:
        logger.warning(f"\nCertificación FALLIDA en {total_duration:.2f}s")
        logger.warning(f"Ver detalles en: {report_output_path}")

    logger.info("=" * 60)

    if not certification:
        raise ValueError(f"Certificación FALLIDA. Ver reporte: {report_output_path}")

    except Exception as e:
        logger.error(f"ERROR: {str(e)}")
        certification = False
        dbutils.jobs.taskValues.set(key="certification", value=certification)
        raise

    finally:
        if 'logger' in locals():
            logger.close()
        MLOpsLogger.cleanup_job_timestamp()

In [0]:
# Detallar meses a certificar en Línea 122.
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType
import pandas as pd
import numpy as np
from pyspark.testing import assertSchemaEqual
pd.set_option('display.max_colwidth', None)

def compare_dataframes(
    df_base,
    df_target,
    cols_to_compare,
    cols_keys,
    threshold=1e-6,
    suffix_base="_base",
    suffix_target="_target"
):
    def suffix_cols(cols, suffix):
        return [F.col(c).alias(c + suffix) if c not in cols_keys else F.col(c) for c in cols]

    # Compare number of records in each dataframe
    df_base_count = df_base.count()
    df_target_count = df_target.count()
    if df_base_count != df_target_count:
        raise ValueError(f"Count mismatch: df_base={df_base_count}, df_target={df_target_count}")

    # Select & rename
    df_b = df_base.select(*suffix_cols(cols_to_compare + cols_keys, suffix_base))
    df_t = df_target.select(*suffix_cols(cols_to_compare + cols_keys, suffix_target))

    # Join
    df = df_b.join(df_t, cols_keys, "outer")

    # Build comparison expressions
    exprs = df.columns.copy()

    for c in cols_to_compare:
        l = F.col(c + suffix_base)
        r = F.col(c + suffix_target)

        ldt = df.select(l).schema[0].dataType
        rdt = df.select(r).schema[0].dataType

        if isinstance(ldt, StringType) and isinstance(rdt, StringType):
            exprs.append(
                F.when(l.isNull() & r.isNull(), 1)
                .when(l == r, 1)
                .otherwise(0)
                .alias(c + "_eq")
            )
        elif isinstance(ldt, NumericType) and isinstance(rdt, NumericType):
            exprs.extend([
                (l - r).alias(c + "_diff"),
                F.when(l.isNull() & r.isNull(), 1)
                .when(F.abs(l - r) <= threshold, 1)
                .otherwise(0)
                .alias(c + "_eq")
            ])

    df_cmp = df.select(*exprs).cache()
    # df_cmp.count()

    # Summary
    eq_cols = [c for c in df_cmp.columns if c.endswith("_eq")]

    ones = df_cmp.agg(*[F.sum(c).alias(c) for c in eq_cols]).withColumn("eq", F.lit(1))
    zeros = df_cmp.agg(*[(F.count("*") - F.sum(c)).alias(c) for c in eq_cols]).withColumn("eq", F.lit(0))

    df_summary = ones.unionByName(zeros).select("eq", *eq_cols)

    pd_summary = (
        df_summary
        .orderBy("eq")
        .toPandas()
        .set_index("eq")
        .T
        .reset_index()
        .rename(columns={"index": "var", 0: "count_0", 1: "count_1"})
        .sort_values("count_0", ascending=False)
    )
    pd_summary['porc_equiv'] = np.divide(pd_summary['count_1'], pd_summary['count_0'] + pd_summary['count_1'])*100
    pd_summary['porc_equiv'] = pd_summary['porc_equiv'].round(6)
    pd_summary['validacion'] = np.where(pd_summary['porc_equiv'] >= 99.0, '✅', '❌')

    return df_cmp, pd_summary


from pyspark.sql.types import StructType

def compare_schemas(schema1: StructType, schema2: StructType):
    fields1 = {f.name: f for f in schema1.fields}
    fields2 = {f.name: f for f in schema2.fields}
    all_cols = set(fields1.keys()).union(fields2.keys())

    diffs = []
    for col in sorted(all_cols):
        f1 = fields1.get(col)
        f2 = fields2.get(col)

        if f1 is None:
            diffs.append(f"+ Column only in target: {col} {f2.dataType}")
        elif f2 is None:
            diffs.append(f"- Column only in base: {col} {f1.dataType}")
        else:
            differences = []
            if f1.dataType != f2.dataType:
                differences.append(f"type: {f1.dataType} != {f2.dataType}")
            if f1.nullable != f2.nullable:
                differences.append(f"nullable: {f1.nullable} != {f2.nullable}")

            if differences:
                diffs.append(f"* Column '{col}' differs -> " + ", ".join(differences))

    return diffs

df_target = spark.table(output_table)
if isinstance(scoring_path_target, str):
    df_target = spark.read.format("delta").table(scoring_path_target) \
        .filter(col("codmes") == mes_vta)
else:
    df_target = scoring_path_target \
        .filter(col("codmes") == mes_vta)
df_target = df_target.withColumn("numscore", F.round(F.col("numscore"), 0))
df_base = spark.table("catalog_cemm_expl_bcp_prod.bcp_expl_007.bcp_scoring_bhv_troncal_cliente_v2_dev")
compare_schemas(df_base.schema, df_target.schema)

df_target = spark.table(output_table)

config_codmes_list = [202510, 202511, 202512]

cols_keys = ['codmes', 'codclaveunicocli']
vars_cols = [c for c in df_target.columns if c not in cols_keys + ["codclavepartycli", "codinternocomputacional"]]
for codmes in config_codmes_list:
    df_base_codmes = df_base.filter(F.col('codmes')==F.lit(codmes))
    df_target_codmes = df_target.filter(F.col('codmes')==F.lit(codmes))
    df_codmes, summary_codmes = compare_dataframes(
        df_base=df_base_codmes,
        df_target=df_target_codmes,
        cols_to_compare=vars_cols,
        cols_keys=cols_keys,
        threshold=1e-6,
        suffix_base="_base",
        suffix_target="_target"
    )
    print('='*60)
    print(codmes)
    print('-'*60)
    display(summary_codmes)